In [ ]:
import yaml as yml
import pandas as pd
import re
from emu_renewal.inputs import BASE_PATH
from emu_renewal.outputs import get_table_df_from_priors_dict, plot_beta_priors
from IPython.display import Markdown

In [ ]:
loaded_priors = yml.safe_load(open(BASE_PATH / "emu_renewal/priors.yml", "r"))

In [ ]:
col_widths = '{tbl-colwidths="[14, 8, 8, 70]"}'
durations_df = get_table_df_from_priors_dict(pd.DataFrame.from_dict(loaded_priors["durations"]))
caption = "\n: Parameters and supporting evidence to time period priors. "
Markdown(durations_df.to_markdown() + caption + col_widths)

In [ ]:
from numpyro import distributions as dist
import numpy as np
from plotly import graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
duration_priors = {
    k: dist.TruncatedNormal(v["mean"], v["sd"], low=1.0, high=v["mean"] * 2.5) 
    for k, v in loaded_priors["durations"].items()
}

In [ ]:
progress_priors = {k: v for k, v in duration_priors.items() if "gen" not in k and "report" not in k}

In [ ]:
progress_fig = make_subplots(2, 1, subplot_titles=["Mean", "SD"])
x_vals = np.linspace(0.0, 30.0, 1000)
for k, v in progress_priors.items():
    row = 1 if "mean" in k else 2
    progress_fig.add_trace(go.Scatter(x=x_vals, y=np.exp(v.log_prob(x_vals)), name=k), row=row, col=1)

In [ ]:
progress_fig.write_image("progress_fig.svg")

In [ ]:
betas_df = get_table_df_from_priors_dict(pd.DataFrame.from_dict(loaded_priors["beta"]))
caption = "\n: Parameters and supporting evidence to beta-distributed priors. "
Markdown(betas_df.to_markdown() + caption + col_widths)

In [ ]:
#| fig-cap: "Illustration of beta prior distributions"
plot_beta_priors(loaded_priors)